<a href="https://colab.research.google.com/github/leeknnn/viktoriiashkorubska-41/blob/master/lab_work_18/lab_work_18.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
from scipy import stats

# Варіант 1: Київ (Регіон: Центр, Середньорічна температура: 9.5, Сезонна амплітуда: 14, Базова вологість: 74)
np.random.seed(42)
city = "Київ"
mean_temp = 9.5
amplitude = 14
hum_base = 74

rows = []
for month in range(1, 13):
    temp = mean_temp + amplitude * np.cos((month - 7) / 12 * 2 * np.pi) + np.random.normal(0, 1.0)
    humidity = hum_base + np.random.normal(0, 5.0)
    if temp < 8:
        opal_days = 30
    elif temp < 12:
        opal_days = 15
    else:
        opal_days = 0

    cost = (
        200
        - 9.0 * temp
        + 0.6 * humidity
        + 2.5 * opal_days
        + np.random.normal(0, 15)
    )
    rows.append({
        "місто": city, "місяць": month,
        "температура": round(temp, 1),
        "вологість": round(humidity, 1),
        "опалювальні_дні": opal_days,
        "витрати_на_опалення": round(cost, 1),
    })
heating = pd.DataFrame(rows)

print("--- Завдання 1: Модель із двома предикторами ---")
X1 = sm.add_constant(heating[["температура", "вологість"]])
y = heating["витрати_на_опалення"]
model1 = sm.OLS(y, X1).fit()
print(model1.summary())

print("\n--- Завдання 3: Перевірка VIF для моделі 1 ---")
for i, col in enumerate(X1.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X1.values, i)
    print(f"VIF для {col}: {vif_val:.2f}")

print("\n--- Завдання 4: Третій предиктор (опалювальні_дні) ---")
X2 = sm.add_constant(heating[["температура", "вологість", "опалювальні_дні"]])
model2 = sm.OLS(y, X2).fit()
print(f"Модель 1 (2 предиктора): R² = {model1.rsquared:.4f}, Adj. R² = {model1.rsquared_adj:.4f}")
print(f"Модель 2 (3 предиктора): R² = {model2.rsquared:.4f}, Adj. R² = {model2.rsquared_adj:.4f}")

# VIF для трипредикторної моделі
for i, col in enumerate(X2.columns):
    if col == "const":
        continue
    vif_val = variance_inflation_factor(X2.values, i)
    print(f"VIF (Модель 2) для {col}: {vif_val:.2f}")

print("\n--- Завдання 5: Діагностика залишків ---")
fitted = model1.fittedvalues
residuals = model1.resid

shapiro_stat, shapiro_p = stats.shapiro(residuals)
print(f"Критерій Шапіро-Вілка для залишків: statistic = {shapiro_stat:.4f}, p-value = {shapiro_p:.4f}")

--- Завдання 1: Модель із двома предикторами ---
                             OLS Regression Results                            
Dep. Variable:     витрати_на_опалення   R-squared:                       0.976
Model:                             OLS   Adj. R-squared:                  0.970
Method:                  Least Squares   F-statistic:                     179.6
Date:                 Wed, 21 Oct 2026   Prob (F-statistic):           5.58e-08
Time:                         13:08:19   Log-Likelihood:                -52.453
No. Observations:                   12   AIC:                             110.9
Df Residuals:                        9   BIC:                             112.4
Df Model:                            2                                         
Covariance Type:             nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

## Відповіді на контрольні питання

1. **Чому sm.add_constant() є обов'язковим кроком перед sm.OLS()?**
   - Метод `sm.OLS()` за замовчуванням виконує регресію через початок координат (без вільних членів). Функція `sm.add_constant()` додає до матриці ознак стовпець одиниць, що дозволяє моделі обчислити вільний член (перетин з віссю Y, або `const`), без якого оцінка нахилів була б суттєво зміщеною.

2. **У чому коефіцієнт множинної регресії відрізняється від коефіцієнта простої регресії?**
   - У простій регресії коефіцієнт показує загальний зв'язок між однією змінною та відгуком, у тому числі враховуючи можливий вплив інших неявних факторів. У множинній регресії кожен коефіцієнт інтерпретується "за інших рівних умов", тобто він вимірює чистий ефект зміни конкретного предиктора за умови, що всі інші змінні в моделі залишаються зафіксованими на незмінному рівні.

3. **Чому звичайний R² не може зменшитися при додаванні нового предиктора?**
   - Звичайний $R^2$ неминуче зростає або залишається незмінним при включенні будь-яких додаткових змінних (навіть випадкових та безглуздих), оскільки кожна нова змінна додає хоча б мінімальне покращення підгонки під наявну вибірку. Саме тому для порівняння моделей використовують скоригований $R^2$ (`Adj. R²`), який вводить штраф за кожну додаткову степінь свободи.

4. **Що означає VIF, що дорівнює 1, і VIF вище 10?**
   - Значення VIF (коефіцієнт інфляції дисперсії) близько 1 вказує на повну відсутність мультиколінеарності (незалежність предиктора від інших). Значення VIF вище 10 сигналізує про критичну мультиколінеарність, коли предиктор сильно лінійно дублює інші змінні в моделі, що різко збільшує стандартні похибки коефіцієнтів і робить їх нестабільними (на практиці варто вилучати або об'єднувати такі змінні).